In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [5]:
print(train.columns.tolist())
print(train.head())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
   id                                             prompt  \
0   1  Pick the best possible answer: What is the rel...   
1   2  What is the estimated redshift of CEERS-93316,...   
2   3  Pick the best possible answer: What is the rea...   
3   4  What is the significance of the redshift-dista...   
4   5  What is the Landau-Lifshitz-Gilbert equation u...   

                                                   A  \
0  For every eigenstate of one Hamiltonian, its p...   
1  Approximately z = 6.0, corresponding to 1 bill...   
2  The sun appears yellowish due to a reflection ...   
3  Observations of the redshift-distance relation...   
4  The Landau-Lifshitz-Gilbert equation is a diff...   

                                                   B  \
0  For every eigenstate of one Hamiltonian, its p...   
1  Approximately z = 16.7, corresponding to 235.8...   
2  The longer wavelengths of light, such as red a...   
3  Observations of the redshift-dist

In [6]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [7]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
# Q1 — Zero-shot score for correct option at row 150

result_q1 = zs(prompt_150, candidate_labels=labels_150)

# Find score for the correct answer text
correct_idx = result_q1['labels'].index(ans_150)
q1_score    = round(result_q1['scores'][correct_idx], 3)

print(f"\nQ1 — All labels and scores:")
for label, score in zip(result_q1['labels'], result_q1['scores']):
    marker = " ← correct" if label == ans_150 else ""
    print(f"  {round(score,3)} | {label[:60]}{marker}")

print(f"\nQ1 Answer: {q1_score}")


Q1 — All labels and scores:
  0.384 | The butterfly effect is the phenomenon that a small change i ← correct
  0.379 | The butterfly effect is the phenomenon that a large change i
  0.093 | The butterfly effect is the phenomenon that a small change i
  0.079 | The butterfly effect is the phenomenon that a small change i
  0.066 | The butterfly effect is the phenomenon that a large change i

Q1 Answer: 0.384


In [9]:
# Q2 — FAISS retrieval rank of true document for row 150

# Embed the prompt
prompt_emb = model.encode([prompt_150]).astype('float32')

# Query FAISS for top k=10
k = 10
distances, retrieved_indices = index.search(prompt_emb, k)
retrieved_indices = retrieved_indices[0]

print(f"Q2 — Top 10 retrieved indices: {retrieved_indices.tolist()}")

# The true document is at index 150 in KB
true_kb_index = 150
rank = None
for r, idx in enumerate(retrieved_indices):
    if idx == true_kb_index:
        rank = r + 1  # 1-indexed
        break

if rank:
    print(f"Q2 Answer: True document found at rank {rank}")
else:
    print("Q2 Answer: True document NOT found in top 10")

Q2 — Top 10 retrieved indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
Q2 Answer: True document found at rank 10


In [10]:
# Q3 — Cross-Encoder reranking of top 10 FAISS results

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Get the 10 documents
docs_10 = [kb[i] for i in retrieved_indices]

# Score each (prompt, doc) pair
pairs     = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort by cross-encoder score descending
ranked_by_ce = np.argsort(ce_scores)[::-1]

print("Q3 — Cross-Encoder ranked results:")
ce_rank = None
for r, pos in enumerate(ranked_by_ce):
    original_kb_idx = retrieved_indices[pos]
    marker = " ← TRUE DOCUMENT" if original_kb_idx == true_kb_index else ""
    print(f"  Rank {r+1}: KB index {original_kb_idx} | score {round(ce_scores[pos],4)}{marker}")
    if original_kb_idx == true_kb_index:
        ce_rank = r + 1

print(f"\nQ3 Answer: Cross-Encoder places true document at rank {ce_rank}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Q3 — Cross-Encoder ranked results:
  Rank 1: KB index 150 | score 4.758500099182129 ← TRUE DOCUMENT
  Rank 2: KB index 1906 | score 4.752600193023682
  Rank 3: KB index 847 | score 4.752600193023682
  Rank 4: KB index 1693 | score 4.752600193023682
  Rank 5: KB index 1269 | score 4.737500190734863
  Rank 6: KB index 1532 | score 4.737500190734863
  Rank 7: KB index 168 | score 4.707200050354004
  Rank 8: KB index 576 | score 4.686999797821045
  Rank 9: KB index 1701 | score 4.660200119018555
  Rank 10: KB index 663 | score 4.660200119018555

Q3 Answer: Cross-Encoder places true document at rank 1


In [12]:
# Q4 — Token count of RAG string 

row_42     = train.iloc[42]
prompt_42  = str(row_42['prompt'])

# Retrieve top k=5 for row 42
prompt_42_emb             = model.encode([prompt_42]).astype('float32')
distances_42, indices_42  = index.search(prompt_42_emb, 5)
indices_42                = indices_42[0]

# Concatenate 5 docs with single space
docs_5         = [kb[i] for i in indices_42]
concatenated   = ' '.join(docs_5)
rag_string_42  = f"Context: {concatenated} Question: {prompt_42}"

print(f"RAG string (first 200 chars): {rag_string_42[:200]}...")

# Tokenize with bert-base-uncased (no truncation)
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens         = bert_tokenizer(rag_string_42, truncation=False)
token_count    = len(tokens['input_ids'])

print(f"\nQ4 Answer: Total tokens = {token_count}")

RAG string (first 200 chars): Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combinati...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Q4 Answer: Total tokens = 216


In [13]:
# Q5 — RAG with TRUE document for row 150

# Get the exact true document from KB
true_doc_150  = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

print(f"RAG string: {rag_string_150}")

# Run zero-shot on augmented string
result_q5     = zs(rag_string_150, candidate_labels=labels_150)

correct_idx_q5 = result_q5['labels'].index(ans_150)
q5_score       = round(result_q5['scores'][correct_idx_q5], 3)

print(f"\nQ5 — All scores after RAG:")
for label, score in zip(result_q5['labels'], result_q5['scores']):
    marker = " ← correct" if label == ans_150 else ""
    print(f"  {round(score,3)} | {label[:60]}{marker}")

print(f"\nQ5 Answer: {q5_score}")
print(f"Change from Q1: {q1_score} → {q5_score} "
      f"({'improved' if q5_score > q1_score else 'decreased'})")

RAG string: Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.

Q5 — All scores after RAG:
  0.989 | The butterfly effect is the phenomenon that a small change i ← correct
  0.004 | The butterfly effect is the phenomenon that a large change i
  0.003 | The butterfly effect is the phenomenon that a small change i
  0.002 | The butterfly effect is the phenomenon that a small change i
  0.002 | The butterfly effect is the phenomenon that a large change i

Q5 Answer: 0.989
Change from Q1: 0.384 → 0.989 (improved)


In [14]:
# Q6 — Adversarial RAG 

bad_doc       = kb[999]
adv_rag_str   = f"Context: {bad_doc} Question: {prompt_150}"

print(f"Adversarial context (KB[999]): {bad_doc[:100]}...")

result_q6      = zs(adv_rag_str, candidate_labels=labels_150)
correct_idx_q6 = result_q6['labels'].index(ans_150)
q6_score       = round(result_q6['scores'][correct_idx_q6], 3)

print(f"\nQ6 — Scores with adversarial context:")
for label, score in zip(result_q6['labels'], result_q6['scores']):
    marker = " ← correct" if label == ans_150 else ""
    print(f"  {round(score,3)} | {label[:60]}{marker}")

print(f"\nQ6 Answer: {q6_score}")
print(f"Score dropped from {q1_score} (no RAG) → {q6_score} (bad RAG)")

Adversarial context (KB[999]): A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts o...

Q6 — Scores with adversarial context:
  0.529 | The butterfly effect is the phenomenon that a small change i ← correct
  0.425 | The butterfly effect is the phenomenon that a large change i
  0.02 | The butterfly effect is the phenomenon that a small change i
  0.019 | The butterfly effect is the phenomenon that a large change i
  0.007 | The butterfly effect is the phenomenon that a small change i

Q6 Answer: 0.529
Score dropped from 0.384 (no RAG) → 0.529 (bad RAG)


In [15]:
# Q7 — Hit Rate for first 100 rows (k=5)

print("Computing hit rate for first 100 rows...")
hits = 0

for i in range(100):
    row        = train.iloc[i]
    prompt     = str(row['prompt'])
    true_ans   = str(row[row['answer']])  # correct answer text

    # Retrieve top 5
    p_emb      = model.encode([prompt]).astype('float32')
    _, idxs    = index.search(p_emb, 5)
    retrieved_docs = [kb[j] for j in idxs[0]]

    # Check if true answer text is in any retrieved doc
    if any(true_ans in doc for doc in retrieved_docs):
        hits += 1

hit_rate = round((hits / 100) * 100, 1)
print(f"\nHits: {hits}/100")
print(f"Q7 Answer: Hit Rate = {hit_rate}%")

Computing hit rate for first 100 rows...

Hits: 73/100
Q7 Answer: Hit Rate = 73.0%


In [16]:
# Q8 — Full RAG Pipeline MAP@3 for first 20 rows

def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0

print("Running full RAG pipeline on first 20 rows...")
map_scores = []

for i in range(20):
    row     = train.iloc[i]
    prompt  = str(row['prompt'])
    options = {opt: str(row[opt]) for opt in ['A','B','C','D','E']}
    correct = row['answer']

    # Step 1 — FAISS retrieve top k=5
    p_emb  = model.encode([prompt]).astype('float32')
    _, idxs = index.search(p_emb, 5)
    docs_5  = [kb[j] for j in idxs[0]]

    # Step 2 — Cross-Encoder rerank, pick best
    pairs    = [[prompt, doc] for doc in docs_5]
    ce_scores_i = cross_encoder.predict(pairs)
    best_doc = docs_5[np.argmax(ce_scores_i)]

    # Step 3 — Build RAG string
    rag_str  = f"Context: {best_doc} Question: {prompt}"

    # Step 4 — Zero-shot classify
    result   = zs(rag_str, candidate_labels=list(options.values()))

    # Step 5 — Map back to letters, compute MAP@3
    val_to_opt = {v: k for k, v in options.items()}
    ranked_opts = [val_to_opt[label] for label in result['labels']]
    score = map_at_3(correct, ranked_opts[:3])
    map_scores.append(score)

    print(f"  Row {i:2d} | Correct: {correct} | "
          f"Predicted: {ranked_opts[:3]} | MAP@3: {score}")

final_map3 = round(sum(map_scores)/len(map_scores), 3)
print(f"\nQ8 Answer: RAG Pipeline MAP@3 = {final_map3}")

Running full RAG pipeline on first 20 rows...
  Row  0 | Correct: B | Predicted: ['B', 'D', 'A'] | MAP@3: 1.0
  Row  1 | Correct: A | Predicted: ['A', 'E', 'C'] | MAP@3: 1.0
  Row  2 | Correct: C | Predicted: ['C', 'D', 'B'] | MAP@3: 1.0
  Row  3 | Correct: B | Predicted: ['B', 'D', 'A'] | MAP@3: 1.0
  Row  4 | Correct: A | Predicted: ['A', 'B', 'C'] | MAP@3: 1.0
  Row  5 | Correct: C | Predicted: ['B', 'C', 'A'] | MAP@3: 0.5
  Row  6 | Correct: E | Predicted: ['E', 'B', 'D'] | MAP@3: 1.0
  Row  7 | Correct: A | Predicted: ['A', 'B', 'C'] | MAP@3: 1.0
  Row  8 | Correct: A | Predicted: ['A', 'C', 'D'] | MAP@3: 1.0
  Row  9 | Correct: A | Predicted: ['A', 'B', 'C'] | MAP@3: 1.0
  Row 10 | Correct: C | Predicted: ['C', 'A', 'D'] | MAP@3: 1.0
  Row 11 | Correct: B | Predicted: ['B', 'C', 'A'] | MAP@3: 1.0
  Row 12 | Correct: D | Predicted: ['D', 'A', 'B'] | MAP@3: 1.0
  Row 13 | Correct: E | Predicted: ['E', 'D', 'B'] | MAP@3: 1.0
  Row 14 | Correct: E | Predicted: ['E', 'A', 'D'] | MAP@3